In [8]:
# notebooks/02_training.ipynb - Enhanced DistilBERT Fine-tuning

import pandas as pd
import numpy as np
import torch
import os
import json
import warnings
from datetime import datetime
from collections import Counter
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
import transformers

# Suppress warnings
warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("🚀 Enhanced DistilBERT Fine-tuning Pipeline")
print("=" * 60)
print(f"Transformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============================================================================
# 1. CONFIGURATION
# ============================================================================

CONFIG = {
    # Paths
    "data_path": "../data/processed/labeled.csv",
    "model_output_dir": "../models/distilbert_finetuned",
    "logs_dir": "../models/logs",
    "results_dir": "../models/results",
    
    # Data parameters
    "test_size": 0.2,
    "val_size": 0.1,  # Additional validation split
    "random_state": 42,
    "max_length": 128,
    
    # Model parameters
    "base_model": "distilbert-base-uncased",
    "dropout_rate": 0.1,
    
    # Training parameters
    "num_epochs": 5,
    "batch_size": 16,
    "eval_batch_size": 32,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,
    
    # Monitoring
    "logging_steps": 50,
    "eval_steps": 200,
    "save_steps": 500,
    "save_total_limit": 3,
    "early_stopping_patience": 3,
    "metric_for_best_model": "eval_f1_weighted",
    "load_best_model_at_end": True
}

# Create directories
for dir_path in [CONFIG["model_output_dir"], CONFIG["logs_dir"], CONFIG["results_dir"]]:
    Path(dir_path).mkdir(parents=True, exist_ok=True)

print(f"\n⚙️  Configuration:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")

# ============================================================================
# 2. DATA LOADING AND EXPLORATION
# ============================================================================

print(f"\n📁 Loading dataset from: {CONFIG['data_path']}")

try:
    df = pd.read_csv(CONFIG["data_path"])
    print(f"✅ Dataset loaded successfully")
except FileNotFoundError:
    print(f"❌ Error: Dataset not found at {CONFIG['data_path']}")
    raise

print(f"   Dataset shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")

# Data validation
required_columns = ["text", "label"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Handle missing values
print(f"\n🧹 Data cleaning:")
initial_len = len(df)
df = df.dropna(subset=required_columns)
df = df[df["text"].str.strip() != ""]  # Remove empty texts
print(f"   Removed {initial_len - len(df)} invalid samples")
print(f"   Final dataset size: {len(df)}")

# Display class distribution
print(f"\n📊 Class distribution:")
class_counts = df["label"].value_counts().sort_index()
print(class_counts)

# Calculate class imbalance ratio
max_count = class_counts.max()
min_count = class_counts.min()
imbalance_ratio = max_count / min_count
print(f"   Imbalance ratio: {imbalance_ratio:.2f}")

if imbalance_ratio > 3:
    print(f"   ⚠️  High class imbalance detected! Consider using class weights.")

# ============================================================================
# 3. LABEL ENCODING AND MAPPING
# ============================================================================

print(f"\n🏷️  Label encoding:")

# Create consistent label mapping
unique_labels = sorted(df["label"].unique())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(unique_labels)

print(f"   Number of classes: {num_labels}")
for label, idx in label2id.items():
    print(f"   {label} → {idx}")

# Apply label encoding
df["label_id"] = df["label"].map(label2id)

# ============================================================================
# 4. DATASET SPLITTING
# ============================================================================

print(f"\n🔄 Dataset splitting:")

# First split: train + temp vs final test
train_temp_texts, test_texts, train_temp_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label_id"].tolist(),
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
    stratify=df["label_id"].tolist()
)

# Second split: train vs validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_temp_texts,
    train_temp_labels,
    test_size=CONFIG["val_size"] / (1 - CONFIG["test_size"]),  # Adjust for remaining data
    random_state=CONFIG["random_state"],
    stratify=train_temp_labels
)

print(f"   Train samples:      {len(train_texts)}")
print(f"   Validation samples: {len(val_texts)}")
print(f"   Test samples:       {len(test_texts)}")

# Verify stratification
print(f"\n   Class distribution across splits:")
for split_name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    split_counts = Counter(labels)
    split_dist = [split_counts.get(i, 0) for i in range(num_labels)]
    print(f"   {split_name:>5}: {split_dist}")

# ============================================================================
# 5. TOKENIZER AND DATASET PREPARATION
# ============================================================================

print(f"\n🔤 Loading tokenizer: {CONFIG['base_model']}")

tokenizer = DistilBertTokenizerFast.from_pretrained(CONFIG["base_model"])

# Add special tokens if needed (for domain-specific data)
# special_tokens = {"additional_special_tokens": ["<PHONE>", "<ADDRESS>", "<BUSINESS>"]}
# tokenizer.add_special_tokens(special_tokens)

print(f"   Tokenizer loaded successfully")
print(f"   Vocabulary size: {len(tokenizer)}")

def create_dataset(texts, labels):
    """Create HuggingFace Dataset from texts and labels"""
    return Dataset.from_dict({
        "text": texts,
        "label": labels
    })

def tokenize_function(examples):
    """Tokenization function for batched processing"""
    return tokenizer(
        examples["text"],
        padding=False,  # Will be handled by DataCollator
        truncation=True,
        max_length=CONFIG["max_length"],
        return_tensors=None
    )

# Create datasets
print(f"\n📦 Creating datasets...")
train_dataset = create_dataset(train_texts, train_labels)
val_dataset = create_dataset(val_texts, val_labels)
test_dataset = create_dataset(test_texts, test_labels)

# Apply tokenization
print(f"   Tokenizing datasets...")
train_dataset = train_dataset.map(
    tokenize_function, 
    batched=True, 
    desc="Tokenizing train dataset"
)
val_dataset = val_dataset.map(
    tokenize_function, 
    batched=True, 
    desc="Tokenizing validation dataset"
)
test_dataset = test_dataset.map(
    tokenize_function, 
    batched=True, 
    desc="Tokenizing test dataset"
)

print(f"✅ Datasets tokenized successfully")

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ============================================================================
# 6. MODEL INITIALIZATION
# ============================================================================

print(f"\n🤖 Loading model: {CONFIG['base_model']}")

# DistilBERT uses different parameter names than BERT
model = DistilBertForSequenceClassification.from_pretrained(
    CONFIG["base_model"],
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dropout=CONFIG["dropout_rate"],  # DistilBERT uses 'dropout' instead of 'hidden_dropout_prob'
    attention_dropout=CONFIG["dropout_rate"]  # DistilBERT uses 'attention_dropout'
)

# If we added special tokens, resize embeddings
# if special_tokens:
#     model.resize_token_embeddings(len(tokenizer))

print(f"✅ Model loaded successfully")
print(f"   Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ============================================================================
# 7. CLASS WEIGHT CALCULATION (for imbalanced data)
# ============================================================================

print(f"\n⚖️  Calculating class weights for imbalanced data...")

# Calculate class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

print(f"   Class weights:")
for i, weight in enumerate(class_weights):
    print(f"   {id2label[i]}: {weight:.3f}")

# Convert to tensor for loss calculation
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# ============================================================================
# 8. CUSTOM TRAINER WITH CLASS WEIGHTS
# ============================================================================

class WeightedTrainer(Trainer):
    """Custom Trainer with class weights for handling imbalanced data"""
    
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Compute loss with class weights, handling different Transformers versions
        """
        # Extract labels from inputs
        labels = inputs.get("labels")
        
        # Remove labels from inputs for model forward pass
        inputs_for_model = {k: v for k, v in inputs.items() if k != "labels"}
        
        # Forward pass
        outputs = model(**inputs_for_model)
        logits = outputs.get("logits")
        
        # Calculate weighted loss if class weights are provided
        if self.class_weights is not None:
            # Move class weights to the same device as the model
            weights = self.class_weights.to(logits.device)
            loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
            loss = loss_fn(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        else:
            # Use default loss calculation
            loss = outputs.loss
        
        return (loss, outputs) if return_outputs else loss

# Alternative simpler approach if the above doesn't work:
class SimpleWeightedTrainer(Trainer):
    """Simpler alternative that modifies the inputs before loss computation"""
    
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Handle different parameter signatures across Transformers versions
        """
        try:
            # Try the new signature first
            return self._compute_loss_new(model, inputs, return_outputs, **kwargs)
        except TypeError:
            # Fall back to old signature
            return self._compute_loss_old(model, inputs, return_outputs)
    
    def _compute_loss_new(self, model, inputs, return_outputs=False, **kwargs):
        """Implementation for newer Transformers versions"""
        labels = inputs.get("labels")
        inputs_for_model = {k: v for k, v in inputs.items() if k != "labels"}
        outputs = model(**inputs_for_model)
        logits = outputs.get("logits")
        
        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
            loss = loss_fn(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        else:
            loss = outputs.loss
        
        return (loss, outputs) if return_outputs else loss
    
    def _compute_loss_old(self, model, inputs, return_outputs=False):
        """Implementation for older Transformers versions"""
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
            loss = loss_fn(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        else:
            loss = outputs.loss
        
        return (loss, outputs) if return_outputs else loss

# ============================================================================
# 9. EVALUATION METRICS
# ============================================================================

def compute_metrics(eval_pred):
    """Comprehensive evaluation metrics"""
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    
    # Calculate various metrics
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1_micro, _ = precision_recall_fscore_support(
        labels, preds, average='micro'
    )
    _, _, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average='macro'
    )
    _, _, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    
    # Per-class F1 scores
    _, _, f1_per_class, _ = precision_recall_fscore_support(
        labels, preds, average=None
    )
    
    metrics = {
        'accuracy': accuracy,
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision': precision,
        'recall': recall
    }
    
    # Add per-class F1 scores
    for i, f1_score in enumerate(f1_per_class):
        metrics[f'f1_class_{id2label[i]}'] = f1_score
    
    return metrics

# ============================================================================
# 10. TRAINING ARGUMENTS
# ============================================================================

print(f"\n⚙️  Setting up training arguments...")

# Check transformers version for compatibility
transformers_version = transformers.__version__
print(f"   Detected transformers version: {transformers_version}")

# Parse version string properly
try:
    from packaging import version
    has_packaging = True
except ImportError:
    has_packaging = False
    print("   ⚠️  'packaging' module not available, using string comparison")

# Ensure save_steps is a multiple of eval_steps for load_best_model_at_end
eval_steps = CONFIG["eval_steps"]
save_steps = CONFIG["save_steps"]

# Adjust save_steps to be a multiple of eval_steps
if CONFIG["load_best_model_at_end"] and save_steps % eval_steps != 0:
    print(f"   ⚠️  Adjusting save_steps from {save_steps} to be multiple of eval_steps {eval_steps}")
    # Find the nearest multiple that's >= the original save_steps
    new_save_steps = ((save_steps + eval_steps - 1) // eval_steps) * eval_steps
    print(f"   ✅ New save_steps: {new_save_steps}")
    save_steps = new_save_steps
else:
    print(f"   ✅ save_steps ({save_steps}) is multiple of eval_steps ({eval_steps})")

# Create training arguments with version-compatible parameters
training_args_dict = {
    # Output and logging
    "output_dir": CONFIG["results_dir"],
    "logging_dir": CONFIG["logs_dir"],
    "logging_steps": CONFIG["logging_steps"],
    
    # Training parameters
    "num_train_epochs": CONFIG["num_epochs"],
    "per_device_train_batch_size": CONFIG["batch_size"],
    "per_device_eval_batch_size": CONFIG["eval_batch_size"],
    "gradient_accumulation_steps": 1,
    
    # Optimization
    "learning_rate": CONFIG["learning_rate"],
    "weight_decay": CONFIG["weight_decay"],
    "warmup_ratio": CONFIG["warmup_ratio"],
    "max_grad_norm": CONFIG["max_grad_norm"],
    
    # Evaluation and saving
    "eval_steps": eval_steps,
    "save_steps": save_steps,
    "save_total_limit": CONFIG["save_total_limit"],
    
    # Model selection
    "load_best_model_at_end": CONFIG["load_best_model_at_end"],
    "metric_for_best_model": CONFIG["metric_for_best_model"],
    "greater_is_better": True,
    
    # Reproducibility
    "seed": CONFIG["random_state"],
    
    # Reporting
    "report_to": [],  # Empty list to disable wandb/tensorboard
}

# Handle parameter name changes for different versions
if has_packaging:
    # Use proper version comparison with packaging module
    if version.parse(transformers_version) >= version.parse("4.38.0"):
        print("   Using parameter names for Transformers >= 4.38.0")
        training_args_dict.update({
            "eval_strategy": "steps",
            "save_strategy": "steps",
        })
    elif version.parse(transformers_version) >= version.parse("4.21.0"):
        print("   Using parameter names for Transformers >= 4.21.0")
        training_args_dict.update({
            "evaluation_strategy": "steps",
            "save_strategy": "steps",
        })
    else:
        print("   Using legacy parameter names")
        training_args_dict.update({
            "evaluate_during_training": True,
        })
else:
    # Fallback: simple string comparison (less reliable)
    version_parts = transformers_version.split('.')
    major = int(version_parts[0])
    minor = int(version_parts[1]) if len(version_parts) > 1 else 0
    
    if major > 4 or (major == 4 and minor >= 38):
        print("   Using parameter names for Transformers >= 4.38.0 (fallback)")
        training_args_dict.update({
            "eval_strategy": "steps",
            "save_strategy": "steps",
        })
    elif major > 4 or (major == 4 and minor >= 21):
        print("   Using parameter names for Transformers >= 4.21.0 (fallback)")
        training_args_dict.update({
            "evaluation_strategy": "steps",
            "save_strategy": "steps",
        })
    else:
        print("   Using legacy parameter names (fallback)")
        training_args_dict.update({
            "evaluate_during_training": True,
        })

# Add hardware optimization parameters if available
try:
    if torch.cuda.is_available():
        training_args_dict["fp16"] = True
        training_args_dict["dataloader_pin_memory"] = True
        training_args_dict["dataloader_num_workers"] = 2
except:
    print("   ⚠️  Some optimization parameters not available in this version")

# Add run name if supported
try:
    training_args_dict["run_name"] = f"distilbert_finetuned_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
except:
    pass

# Final parameter validation and cleanup
# Remove any None values that might cause issues
training_args_dict = {k: v for k, v in training_args_dict.items() if v is not None}

print(f"   Final training arguments:")
for key, value in training_args_dict.items():
    print(f"     {key}: {value}")

training_args = TrainingArguments(**training_args_dict)

print(f"✅ Training arguments configured")
# ============================================================================
# 11. TRAINER INITIALIZATION
# ============================================================================

print(f"\n🏋️  Initializing trainer...")

# Try the simple approach first
trainer = SimpleWeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=CONFIG["early_stopping_patience"]
        )
    ]
)

print(f"✅ Trainer initialized successfully")

# ============================================================================
# 12. TRAINING
# ============================================================================

print(f"\n🚀 Starting training...")
print(f"   Epochs: {CONFIG['num_epochs']}")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Learning rate: {CONFIG['learning_rate']}")
print(f"   Using class weights: {'Yes' if class_weights_tensor is not None else 'No'}")

# Save configuration before training
config_save_path = os.path.join(CONFIG["model_output_dir"], "training_config.json")
with open(config_save_path, 'w') as f:
    # Convert numpy types to native Python types for JSON serialization
    config_to_save = CONFIG.copy()
    config_to_save['label2id'] = label2id
    config_to_save['id2label'] = id2label
    config_to_save['class_weights'] = class_weights.tolist()
    json.dump(config_to_save, f, indent=2)

print(f"   Configuration saved to: {config_save_path}")

# Start training
start_time = datetime.now()
print(f"   Training started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

try:
    trainer.train()
    training_successful = True
    print(f"✅ Training completed successfully!")
except Exception as e:
    training_successful = False
    print(f"❌ Training failed with error: {str(e)}")
    raise

end_time = datetime.now()
training_duration = end_time - start_time
print(f"   Training ended at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Total training time: {training_duration}")

# ============================================================================
# 13. FINAL EVALUATION
# ============================================================================

if training_successful:
    print(f"\n📊 Final evaluation on validation set...")
    
    # Evaluate on validation set
    eval_results = trainer.evaluate()
    
    print(f"   Final Validation Results:")
    for metric, value in eval_results.items():
        if metric.startswith('eval_'):
            metric_name = metric.replace('eval_', '')
            print(f"   {metric_name:>15}: {value:.4f}")

    # ========================================================================
    # 14. MODEL SAVING
    # ========================================================================
    
    print(f"\n💾 Saving model and tokenizer...")
    
    # Save the model and tokenizer
    trainer.save_model(CONFIG["model_output_dir"])
    tokenizer.save_pretrained(CONFIG["model_output_dir"])
    
    print(f"✅ Model saved to: {CONFIG['model_output_dir']}")
    
    # Save training history
    history_path = os.path.join(CONFIG["model_output_dir"], "training_history.json")
    with open(history_path, 'w') as f:
        json.dump({
            'training_duration': str(training_duration),
            'final_eval_results': eval_results,
            'training_args': training_args.to_dict()
        }, f, indent=2)
    
    print(f"✅ Training history saved to: {history_path}")
    
    # ========================================================================
    # 15. QUICK TEST PREDICTION
    # ========================================================================
    
    print(f"\n🔮 Testing model with sample predictions...")
    
    # Load the saved model for testing
    from transformers import pipeline
    
    classifier = pipeline(
        "text-classification",
        model=CONFIG["model_output_dir"],
        tokenizer=CONFIG["model_output_dir"],
        device=0 if torch.cuda.is_available() else -1
    )
    
    # Test samples (you can customize these)
    test_samples = [
        "McDonald's Restaurant",
        "123 Main Street, New York, NY 10001",
        "Men's clothing store",
        "+1-555-123-4567"
    ]
    
    print(f"   Sample predictions:")
    for i, text in enumerate(test_samples, 1):
        try:
            result = classifier(text)
            predicted_label = result[0]['label']
            confidence = result[0]['score']
            print(f"   {i}. '{text}' → {predicted_label} ({confidence:.3f})")
        except Exception as e:
            print(f"   {i}. '{text}' → Error: {str(e)}")

print(f"\n🎉 Training pipeline completed successfully!")
print(f"📁 Model artifacts saved in: {CONFIG['model_output_dir']}")
print(f"📊 Training logs available in: {CONFIG['logs_dir']}")

🚀 Enhanced DistilBERT Fine-tuning Pipeline
Transformers version: 4.56.0
PyTorch version: 2.8.0+cpu
CUDA available: False

⚙️  Configuration:
   data_path: ../data/processed/labeled.csv
   model_output_dir: ../models/distilbert_finetuned
   logs_dir: ../models/logs
   results_dir: ../models/results
   test_size: 0.2
   val_size: 0.1
   random_state: 42
   max_length: 128
   base_model: distilbert-base-uncased
   dropout_rate: 0.1
   num_epochs: 5
   batch_size: 16
   eval_batch_size: 32
   learning_rate: 2e-05
   weight_decay: 0.01
   warmup_ratio: 0.1
   max_grad_norm: 1.0
   logging_steps: 50
   eval_steps: 200
   save_steps: 500
   save_total_limit: 3
   early_stopping_patience: 3
   metric_for_best_model: eval_f1_weighted
   load_best_model_at_end: True

📁 Loading dataset from: ../data/processed/labeled.csv
✅ Dataset loaded successfully
   Dataset shape: (10484, 2)
   Columns: ['text', 'label']

🧹 Data cleaning:
   Removed 0 invalid samples
   Final dataset size: 10484

📊 Class dist

Tokenizing test dataset: 100%|██████████| 2097/2097 [00:00<00:00, 53371.78 examples/s]


✅ Datasets tokenized successfully

🤖 Loading model: distilbert-base-uncased
✅ Model loaded successfully
   Model parameters: 66,956,548
   Trainable parameters: 66,956,548

⚖️  Calculating class weights for imbalanced data...
   Class weights:
   address: 1.000
   category: 1.000
   name: 1.000
   phone: 1.000

⚙️  Setting up training arguments...
   Detected transformers version: 4.56.0
   ⚠️  Adjusting save_steps from 500 to be multiple of eval_steps 200
   ✅ New save_steps: 600
   Using parameter names for Transformers >= 4.38.0
   Final training arguments:
     output_dir: ../models/results
     logging_dir: ../models/logs
     logging_steps: 50
     num_train_epochs: 5
     per_device_train_batch_size: 16
     per_device_eval_batch_size: 32
     gradient_accumulation_steps: 1
     learning_rate: 2e-05
     weight_decay: 0.01
     warmup_ratio: 0.1
     max_grad_norm: 1.0
     eval_steps: 200
     save_steps: 600
     save_total_limit: 3
     load_best_model_at_end: True
     metri